In [153]:
import os, sys, numpy
from typing import Any
from pandas import Series, DataFrame, Timestamp, Timedelta
from pandas import read_pickle, read_csv, concat
from pandas import Index, MultiIndex, DatetimeIndex
from tqdm import tqdm

tests = dict[str, Any]()
for fn in os.listdir("../tests/"):
    if fn.endswith(".pickle"):
        key = fn[: 14]
        if key not in tests:
            tests[key] = list[str]()
        list.append(tests[key], fn)

print("Available tests:")
for key, files in tests.items():
    print(f" >> {key}: ({len(files)} files) ... {files}")
file_keys = ["ticks_p", "candles_p", "ticks_b", "candles_b"]

Available tests:
 >> 20260514040334: (4 files) ... ['20260514040334_candles_b.pickle', '20260514040334_candles_p.pickle', '20260514040334_ticks_b.pickle', '20260514040334_ticks_p.pickle']
 >> 20260514040608: (4 files) ... ['20260514040608_candles_b.pickle', '20260514040608_candles_p.pickle', '20260514040608_ticks_b.pickle', '20260514040608_ticks_p.pickle']
 >> 20260514040716: (4 files) ... ['20260514040716_candles_b.pickle', '20260514040716_candles_p.pickle', '20260514040716_ticks_b.pickle', '20260514040716_ticks_p.pickle']
 >> 20260514040804: (4 files) ... ['20260514040804_candles_b.pickle', '20260514040804_candles_p.pickle', '20260514040804_ticks_b.pickle', '20260514040804_ticks_p.pickle']
 >> 20260514040839: (4 files) ... ['20260514040839_candles_b.pickle', '20260514040839_candles_p.pickle', '20260514040839_ticks_b.pickle', '20260514040839_ticks_p.pickle']
 >> 20260514040906: (4 files) ... ['20260514040906_candles_b.pickle', '20260514040906_candles_p.pickle', '20260514040906_ticks_b

In [168]:
side = "b"
columns = Index([*"ohlc"])
stats = dict()
ccount = dict()
iterator = tqdm(tests.items())
for key, files in iterator:
    ccount[key] = dict()
    iterator.set_description(f"Processing {key}...")
    dft = read_pickle(f"../tests/{key}_ticks_b.pickle")
    dfc = read_pickle(f"../tests/{key}_candles_b.pickle")
    dfp = read_pickle(f"../tests/{key}_candles_p.pickle")
    dft = dft.reset_index(["venue"], drop = True)
    dfc = dfc.reset_index(["venue"], drop = True)
    dfp = dfp.reset_index(["venue"], drop = True)
    dft = dft.reset_index("time")
    dft["time"] = dft["time"].dt.floor("1s")
    dft = dft.groupby(["symbol", "time"])["p" + side].apply(list)
    dfc = dfc[[*(columns + side), "volume"]]
    dfp = dfp[[*(columns + side), "volume"]]
    dfc.columns, dfp.columns = [*columns, "v"], [*columns, "v"]
    dfc = dfc.rename_axis("field", axis = "columns")
    dfp = dfp.rename_axis("field", axis = "columns")
    
    for tf in dfp.index.get_level_values("tf").unique():
        nc = dfc.query("tf == @tf").shape[0]
        np = dfp.query("tf == @tf").shape[0]
        ccount[key][tf] = {"b": nc, "p": np, "dt": Timedelta(
                tf[1 :] + tf[0].lower().replace("m", "min"))}
    dfc = dfc.xs("S1", level = "tf", drop_level = True)
    dfp = dfp.xs("S1", level = "tf", drop_level = True)

    comp = DataFrame.merge(dfc.stack("field").rename("b"), dfp.stack("field").rename("p"),
            left_index = True, right_index = True, how = "outer").sort_index()#.dropna()
    stat: Series = comp["b"] / comp["p"] - 1
    stat = stat.groupby(["symbol", "field"]).describe()
    stats[key] = stat

stats = concat(stats, names = ["test", "symbol", "field"])
ccount = DataFrame.from_dict(ccount, orient = "index")
ccount = ccount.stack().apply(Series)
ccount = ccount.rename_axis(["key", "tf"])
ccount["diff"] = ccount["p"] - ccount["b"]
ccount["pat"] = ccount.apply("{0[b]}/{0[p]}".format, axis = "columns")
# ccount["pat"].unstack("tf").sort_index().transpose()

Processing 20260514041113...: 100%|██████████| 7/7 [00:32<00:00,  4.60s/it]


In [171]:
df = comp.unstack("field")
df: DataFrame = df.loc["GVFC"]
#df = df["b"] - df["p"]
df = df.sort_index()
df.to_csv("gvfc.csv")

In [214]:
key = "20260514_065846-20260514_080136"
dft = read_csv(f"../logs/{key}_ticks.csv")
dfc = read_csv(f"../logs/{key}_candles.csv")
dfc = dfc.drop(columns = ["venue", "symbol"]).set_index("time")
dft = dft.drop(columns = ["venue", "symbol"]).set_index("time")
dfc.index = DatetimeIndex(dfc.index)
dft.index = DatetimeIndex(dft.index)
tfs = dict.fromkeys(dfc["tf"].unique())
for tf in tfs:
    tfd = tf[1:] + " " + str.lower(tf[0])
    tfs[tf] = Timedelta(str.replace(tfd, "m", "min"))
dfc = dfc.set_index("tf", append = True).swaplevel()
dfc = dfc.sort_index()
dfp = dict.fromkeys(tfs)
for tf, tfd in tfs.items():
    df = dft.copy()[["pa", "pb"]]
    df.index = df.index.floor(tfd)
    dfv = df.groupby("time")["pa"].count().rename("volume")
    df = df.groupby("time").ohlc()
    df.columns = df.columns.map(lambda c: c[1][0] + c[0][-1])
    df = DataFrame.merge(dfv, df, how = "outer",
        left_index = True, right_index = True)
    dfp[tf] = df

dfp = concat(dfp, names = ["tf", "time"]).sort_index()
dfc = dfc.rename_axis("field", axis = "columns").sort_index()
dfp = dfp.rename_axis("field", axis = "columns").sort_index()
df = DataFrame.merge(self = dfc.stack("field").rename("b"),
                     right = dfp.stack("field").rename("p"),
                     left_index = True, right_index = True,
                     how = "outer")
df2 = df.dropna(subset = ["b"])
df2: Series = (df2["b"] - df2["p"]).abs()
stats = df2.groupby(["field", "tf"]).describe()
print(stats.to_string())

             count  mean  std  min  25%  50%  75%  max
field  tf                                             
ca     H1      1.0   0.0  NaN  0.0  0.0  0.0  0.0  0.0
       M1     60.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M10     6.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M12     5.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M15     4.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M2     30.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M20     3.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M3     20.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M30     2.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M4     15.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M5     12.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       M6     10.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       S1   3600.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       S10   360.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       S12   300.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       S15   240.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
       S2 